In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import ticker
from matplotlib.lines import Line2D
from pathlib import Path

from mol_faith.notebooks.results_analysis.shared import load_raw_gt_data
from mol_faith.notebooks.results_analysis.shared import get_target_colors, map_explanation_method_names

In [ ]:
benzene_target_base_path = '../../../results/benzene_target'
benzene_raw_gt_data_df = load_raw_gt_data(base_dir=benzene_target_base_path, target_name='benzene')

conj_sys_target_base_path = '../../../results/conjugated_system_target'
conjugated_system_raw_gt_data_df = load_raw_gt_data(base_dir=conj_sys_target_base_path, target_name='conjugated_system')

hbond_donor_target_base_path = '../../../results/hbond_donor'
hbond_donor_raw_gt_data_df = load_raw_gt_data(base_dir=hbond_donor_target_base_path, target_name='hbond_donor')

In [ ]:
def create_grouped_boxplot(target_dataframes,
                           value_col,
                           group_col,
                           figsize=(7, 3),
                           save_plot=False,
                           output_dir=None,
                           linewidth=2,
                           box_width=0.18,
                           gap=0.1,
                           font_scale=1,
                           show_minmax=True,
                           minmax_marker='D',
                           minmax_size=30,
                           plot_linewidth=0.5,
                           labelsize=9,
                           tick_label_size=8,
                           tick_width=0.5,
                           tick_length=2,
                           reference_fig=None):
    if not target_dataframes:
        print("No data provided!")
        return None

    combined_data = []
    for target, df in target_dataframes.items():
        if df.empty:
            continue
        df_copy = df.copy()
        df_copy['target'] = target
        df_copy = map_explanation_method_names(df_copy)
        combined_data.append(df_copy)

    if not combined_data:
        print("No valid data found!")
        return None
    full_df = pd.concat(combined_data, ignore_index=True)

    if group_col == 'model':
        label_order = ['Transformer', 'CNN', 'GAT', 'GCN', 'GIN']
    else:
        label_order = ['IG', 'SHAP', 'KernelSHAP', 'LIME', 'GradCAM', 'Guided GradCAM', 'GNNExplainer', 'GraphMask']

    targets = list(target_dataframes.keys())

    print(f"Creating seaborn grouped boxplot with {len(label_order)} groups and {len(targets)} targets per group")

    fig, ax = plt.subplots(figsize=figsize)

    sns.set_style("whitegrid", {"axes.grid": False, 'xtick.bottom': True, 'ytick.left': True, 'font': 'Arial'})
    sns.set_context("paper")

    target_colors = get_target_colors()

    box_plot = sns.boxplot(
        data=full_df,
        x=group_col,
        y=value_col,
        hue='target',
        order=label_order,
        linewidth=linewidth,
        ax=ax,
        palette=target_colors,
        width=box_width,
        showfliers=False,
        gap=gap,
        dodge=True,
        legend=False
    )

    ax.axhline(y=0, color='grey', linestyle='-', linewidth=plot_linewidth, alpha=0.5, zorder=1)

    if show_minmax:
        minmax_data = []

        for method in label_order:
            method_data = full_df[full_df[group_col] == method]
            for target in targets:
                target_data = method_data[method_data['target'] == target][value_col].dropna()
                if len(target_data) > 0:
                    min_val = target_data.min()
                    max_val = target_data.max()

                    minmax_data.append({
                        group_col: method,
                        'target': target,
                        'min_val': min_val,
                        'max_val': max_val
                    })

        minmax_df = pd.DataFrame(minmax_data)

        method_positions = list(range(len(label_order)))
        n_targets = len(targets)

        if n_targets > 1:
            group_width = 0.8 / n_targets
            offsets = np.linspace(-0.461 + group_width / 2, 0.461 - group_width / 2, n_targets)
        else:
            offsets = [0]

        for _, row in minmax_df.iterrows():
            method_idx = label_order.index(row[group_col])
            target_idx = targets.index(row['target'])
            x_pos = method_idx + offsets[target_idx] * (1 - gap)

            color = target_colors[row['target']]

            ax.scatter(x_pos, row['min_val'],
                       marker=minmax_marker,
                       s=minmax_size,
                       color=color,
                       edgecolor='black',
                       linewidth=1,
                       alpha=0.8,
                       zorder=10)

            ax.scatter(x_pos, row['max_val'],
                       marker=minmax_marker,
                       s=minmax_size,
                       color=color,
                       edgecolor='black',
                       linewidth=1,
                       alpha=0.8,
                       zorder=10)

    x_label = 'Model' if group_col == 'model' else 'xAI Method'
    ax.set_xlabel(x_label, fontsize=labelsize)
    ax.set_ylabel('Fid+', fontsize=labelsize)
    ax.tick_params(axis='y', labelsize=tick_label_size, width=tick_width, length=tick_length)
    ax.tick_params(axis='x', labelsize=tick_label_size, width=tick_width, length=tick_length)
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.2f'))

    if group_col != 'model':
        if reference_fig is not None:
            ref_ax = reference_fig.get_axes()[0]
            ref_yticks = ref_ax.get_yticks()
            ref_ylim = ref_ax.get_ylim()
            visible_ticks = [tick for tick in ref_yticks if ref_ylim[0] <= tick <= ref_ylim[1]]
            ax.set_yticks(visible_ticks)
            ax.set_ylim(ref_ylim)
        else:
            ax.set_yticks([0.12, 0.08, 0.05, 0.02, 0.00, -0.02, -0.05, -0.08])

    for spine in ax.spines.values():
        spine.set_linewidth(plot_linewidth)

    handles, labels = ax.get_legend_handles_labels()

    if show_minmax:
        minmax_handle = Line2D([0], [0],
                               marker=minmax_marker,
                               color='gray',
                               markerfacecolor='gray',
                               markeredgecolor='black',
                               markersize=8,
                               linewidth=0,
                               alpha=0.8)
        handles.append(minmax_handle)
        labels.append('Min/Max')

    # Move legend outside plot area with updated handles and labels
    # ax.legend(handles=handles, labels=labels,
    #           loc='center left', bbox_to_anchor=(1, 0.5), fontsize=11,
    #           title='Target')

    plt.tight_layout()

    if group_col == 'model':
        saving_prefix = 'model'
    else:
        saving_prefix = 'method'

    if save_plot and output_dir:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        plt.savefig(Path(output_dir) / f'{saving_prefix}_grouped_boxplot.png', dpi=450, bbox_inches='tight')
        plt.savefig(Path(output_dir) / f'{saving_prefix}_grouped_boxplot.svg', format='svg', dpi=450,
                    bbox_inches='tight')

    plt.show()
    return fig

In [ ]:
target_data = {
    'Benzene': benzene_raw_gt_data_df,
    'Conjugated System': conjugated_system_raw_gt_data_df,
    'Hbond Donor': hbond_donor_raw_gt_data_df
}

In [ ]:
output_dir = './'

model_fig = create_grouped_boxplot(target_data,
                                   value_col='metadata_target_fraction',
                                   group_col='model',
                                   figsize=(3, 2.5),
                                   save_plot=True,
                                   box_width=0.8,
                                   gap=0.09,
                                   linewidth=0.5,
                                   plot_linewidth=0.5,
                                   show_minmax=False,
                                   labelsize=7,
                                   tick_label_size=7,
                                   minmax_size=12,
                                   output_dir=output_dir)

model_fig = create_grouped_boxplot(target_data,
                                   value_col='metadata_target_fraction',
                                   group_col='explanation_method',
                                   figsize=(3, 2.5),
                                   save_plot=True,
                                   box_width=0.8,
                                   gap=0.09,
                                   linewidth=0.5,
                                   plot_linewidth=0.5,
                                   show_minmax=False,
                                   labelsize=7,
                                   tick_label_size=7,
                                   minmax_size=12,
                                   output_dir=output_dir,
                                   reference_fig=model_fig)